In [6]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
import equinox as eqx
import esm  # pip install fair-esm==2.0.0
import esm2quinox
import jax.random as jr
import jax.random as jrandom
import optax  # pip install optax
import jax
from functions.model import stripped_PREDICTOR
import transformers
import jax.numpy as jnp
from transformers import AutoTokenizer, AutoModel
import flax


In [7]:
flax

<module 'flax' from '/home/kunzj/miniforge3/envs/BindCraft_modified/lib/python3.10/site-packages/flax/__init__.py'>

In [33]:
a =tokenizer('AAASSSS')['input_ids']
b =tokenizer('AAASSSS')['attention_mask']

In [37]:
import numpy as np

In [44]:
inputs

{'input_ids': tensor([[ 0, 20,  5,  7,  4, 15, 16,  2,  1,  1],
         [ 0, 20,  6, 21, 22, 11,  9,  4,  8,  2]]),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
         [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [46]:
a= model(**inputs)['pooler_output']

In [54]:
a.detach().numpy()

array([[ 0.23471053, -0.38913912, -0.12059745, ..., -0.06329029,
        -0.10155652, -0.1395859 ],
       [ 0.21034375, -0.3897177 , -0.11860169, ..., -0.08459758,
        -0.09402288, -0.13095453]], shape=(2, 1280), dtype=float32)

In [ ]:
np.mean(a.detach().numpy())

np.float32(0.00066909683)

: 

In [39]:
model(a,b)

AttributeError: 'list' object has no attribute 'ne'

In [18]:
model(inp,labels=labels)

AttributeError: 

In [2]:
target_str =['QPRGGGPTSSEQIMKTGALLLQGFIQDRAGRMGGEAPELALDPVPQDASTKKLSECLKRIGDELDSNMELQRMIAAVDTDSPREVFFRVAADMFSDGNFNWGRVVALFYFASKLVLKALCTKVPELIRTIMGWTLDFLRERLLGWIQDQGGWDGLLSYFG']
x_prot = esm2quinox.tokenise(target_str)

In [3]:
# generating keys
model_key, call_key = jr.split(jrandom.PRNGKey(0), 2)

# initializing models
torch_model, _ = esm.pretrained.esm2_t30_150M_UR50D()
model_esm2 = esm2quinox.from_torch(torch_model)
model_aff, model_state = eqx.nn.make_with_state(stripped_PREDICTOR)(
    model=model_esm2, key=model_key
)

# initializing optimizer
optim = optax.adam(learning_rate=0.001)
opt_state = optim.init(eqx.filter(model_aff, eqx.is_inexact_array))
target_str =['QPRGGGPTSSEQIMKTGALLLQGFIQDRAGRMGGEAPELALDPVPQDASTKKLSECLKRIGDELDSNMELQRMIAAVDTDSPREVFFRVAADMFSDGNFNWGRVVALFYFASKLVLKALCTKVPELIRTIMGWTLDFLRERLLGWIQDQGGWDGLLSYFG']
x_prot = esm2quinox.tokenise(target_str)
test_key = model_key[0]

inference_model = eqx.nn.inference_mode(model_aff)
inference_model = eqx.Partial(inference_model, state=model_state)

# load best inference pretrained model
#best_model_aff = eqx.tree_deserialise_leaves('/home/kunzj/BindCraft_uva_internship/surr_model/params/model_inference_second_training_set.eqx',inference_model)
#inference_model(x_prot.squeeze(),x_prot.squeeze(),key=test_key)

W1127 00:41:23.811880  205762 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.
W1127 00:41:23.813973  205763 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.
W1127 00:41:23.818642  205570 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.
W1127 00:41:23.820133  205570 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.


In [ ]:
model_esm2(x_prot.squeeze())

TypeError: ESM2.__call__() got an unexpected keyword argument 'seq_len'

In [7]:
import jax.tree_util as jtu

In [20]:
filter_spec = jtu.tree_map(lambda _: False, best_model_aff)

_,static_model = eqx.partition(best_model_aff, filter_spec)

In [ ]:
_(x_prot.squeeze(),x_prot.squeeze(),key=test_key)

TypeError: unsupported operand type(s) for -: 'int' and 'NoneType'

: 

In [22]:
_

Partial(
  func=stripped_PREDICTOR(
    esm2=ESM2(
      num_layers=30,
      embed_size=640,
      num_heads=20,
      token_dropout=True,
      layers=TransformerLayer(
        embed_size=640,
        hidden_size=2560,
        num_heads=20,
        attn=MultiheadAttention(
          query_proj=Linear(
            weight=None,
            bias=None,
            in_features=640,
            out_features=640,
            use_bias=True
          ),
          key_proj=Linear(
            weight=None,
            bias=None,
            in_features=640,
            out_features=640,
            use_bias=True
          ),
          value_proj=Linear(
            weight=None,
            bias=None,
            in_features=640,
            out_features=640,
            use_bias=True
          ),
          output_proj=Linear(
            weight=None,
            bias=None,
            in_features=640,
            out_features=640,
            use_bias=True
          ),
          dropout=Dropout

In [16]:
t = eqx.combine(_,static_model)

In [10]:
x_prot =jnp.array([ 0, 16, 14, 10,  6,  6,  6, 14, 11,  8,  8,  9, 16, 12, 20, 15,
    11,  6,  5,  4,  4,  4, 16,  6, 18, 12, 16, 13, 10,  5,  6, 10,
    20,  6,  6,  9,  5, 14,  9,  4,  5,  4, 13, 14,  7, 14, 16, 13,
    5,  8, 11, 15, 15,  4,  8,  9, 23,  4, 15, 10, 12,  6, 13,  9,
    4, 13,  8, 17, 20,  9,  4, 16, 10, 20, 12,  5,  5,  7, 13, 11,
    13,  8, 14, 10,  9,  7, 18, 18, 10,  7,  5,  5, 13, 20, 18,  8,
    13,  6, 17, 18, 17, 22,  6, 10,  7,  7,  5,  4, 18, 19, 18,  5,
    8, 15,  4,  7,  4, 15,  5,  4, 23, 11, 15,  7, 14,  9,  4, 12,
    10, 11, 12, 20,  6, 22, 11,  4, 13, 18,  4, 10,  9, 10,  4,  4,
    6, 22, 12, 16, 13, 16,  6,  6, 22, 13,  6,  4,  4,  8, 19, 18,
    6,  2])

In [12]:
model_esm2(x_prot)

ESM2Result(hidden=f32[162,640], logits=f32[162,33])